# Data Cleaning -- Training Dataset
**This notebook prepares Flatiron Health data for patients with advanced head and neck cancer in anticipation of training a gradient-boosted survival model to predict mortality from the start of first-line treatment. Specifically, it processes and cleans the training cohort. Prior to data cleaning, the dataset is randomly split into training (80%) and testing (20%) subsets.**

## Import packages

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from flatiron_cleaner import DataProcessorHeadNeck, merge_dataframes

## Split into train and test sets

In [2]:
df = pd.read_csv('../data/LineOfTherapy.csv')

In [3]:
df = (
    df
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
    [['PatientID', 'StartDate']]
)

In [4]:
df.shape

(8746, 2)

In [5]:
processor = DataProcessorHeadNeck()

In [6]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           drop_dates = False)

2025-11-28 21:43:17,617 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (7857, 2) and unique PatientIDs: 7857
2025-11-28 21:43:17,627 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (8746, 3) and unique PatientIDs: 8746
2025-11-28 21:43:18,010 - INFO - The following columns ['last_visit_date'] are used to calculate the last EHR date
2025-11-28 21:43:18,017 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (8746, 6) and unique PatientIDs: 8746. There are 0 out of 8746 patients with missing duration values


In [7]:
df = pd.merge(df, mortality_df[['PatientID', 'event']], on = 'PatientID', how = 'left')

In [8]:
df.shape

(8746, 3)

In [9]:
# Stratify split on event 
train, test = train_test_split(
    df,
    test_size = 0.2,
    stratify = df['event'],  
    random_state = 42
)

In [10]:
train.shape

(6996, 3)

In [11]:
test.shape

(1750, 3)

In [12]:
train[['PatientID']].to_csv('../outputs/train_patient_ids.csv', index = False)
test[['PatientID']].to_csv('../outputs/test_patient_ids.csv', index = False)

## Clean CSV files 

In [13]:
train_ids = pd.read_csv('../outputs/train_patient_ids.csv')

In [14]:
train_ids = train_ids.PatientID.to_list()

In [15]:
index_date_df = df[df.PatientID.isin(train_ids)]

In [16]:
index_date_df.shape

(6996, 3)

In [17]:
index_date_df = index_date_df[['PatientID', 'StartDate']]

### Process Enhanced_AdvHeadNeck.csv

In [18]:
enhanced_df = processor.process_enhanced(file_path = '../data/Enhanced_AdvHeadNeck.csv',
                                         index_date_df = index_date_df,
                                         index_date_column = 'StartDate',
                                         drop_dates = False)

2025-11-28 21:43:18,097 - INFO - Successfully read Enhanced_AdvHeadNeck.csv file with shape: (11347, 15) and unique PatientIDs: 11347
2025-11-28 21:43:18,103 - INFO - Successfully filtered Enhanced_AdvHeadNeck.csv file with shape: (6996, 16) and unique PatientIDs: 6996
2025-11-28 21:43:18,119 - INFO - Successfully processed Enhanced_AdvHeadNeck.csv file with final shape: (6996, 20) and unique PatientIDs: 6996


In [19]:
enhanced_df['days_adv_to_treatment'] = (enhanced_df['imported_StartDate'] - enhanced_df['AdvancedDiagnosisDate']).dt.days
enhanced_df['days_adv_to_treatment'] = np.where(enhanced_df['days_adv_to_treatment'] < 0 , 0, enhanced_df['days_adv_to_treatment'])

In [20]:
enhanced_df['days_diagnosis_to_adv'] = np.where(enhanced_df['days_diagnosis_to_adv'] < 0 , 0, enhanced_df['days_diagnosis_to_adv'])
enhanced_df['days_diagnosis_to_adv'] = enhanced_df['days_diagnosis_to_adv'].fillna(0)

In [21]:
enhanced_df.GroupStage_mod.value_counts(dropna = False)

GroupStage_mod
IV_locoregional    2941
unknown             966
IV_metastatic       947
III                 938
II                  583
I                   434
IV_NOS              187
Name: count, dtype: int64

In [22]:
enhanced_df['GroupStage_mod'] = enhanced_df["GroupStage_mod"].map({
    '0': 0, 
    'I': 1,
    'II': 2,
    'III': 3,
    'IV_NOS': 4,
    'IV_locoregional': 4,
    'IV_metastatic': 5, 
    'unknown': np.nan
})

enhanced_df['GroupStage_mod_na'] = np.where(enhanced_df['GroupStage_mod'].isna(), 1, 0)

# impute 4 since stage IV (locoregional) is most common
enhanced_df['GroupStage_mod'] = enhanced_df['GroupStage_mod'].fillna(4)

In [23]:
enhanced_df.SmokingStatus.value_counts(dropna = False, normalize = True)

SmokingStatus
History of smoking        0.783877
No history of smoking     0.211549
Unknown/not documented    0.004574
Name: proportion, dtype: float64

In [24]:
# impute unknown as smokers since most common
enhanced_df['SmokingStatus'] = (
    enhanced_df['SmokingStatus']
    .map({'History of smoking': 1, 
          'No history of smoking': 0,
          'Unknown/not documented': 1})
    .astype('Int64')
)

In [25]:
enhanced_df.HPVStatus_mod.value_counts(dropna = False)

HPVStatus_mod
negative    2439
NaN         2241
positive    2173
unknown      143
Name: count, dtype: int64

In [26]:
enhanced_df['HPVStatus_mod'] = enhanced_df['HPVStatus_mod'].fillna('unknown') 

In [27]:
enhanced_df = enhanced_df.drop(columns = ['DiagnosisDate', 
                                          'AdvancedDiagnosisDate', 
                                          'imported_StartDate',
                                          'FirstLocalRecurDate',
                                          'FirstDistantRecurDate',
                                          'PrimarySurgeryDate',
                                          'PrimaryRadiationDate'])

### Process Demographics.csv 

In [28]:
demographics_df = processor.process_demographics(file_path = '../data/Demographics.csv',
                                                 index_date_df = index_date_df,
                                                 index_date_column = 'StartDate')

2025-11-28 21:43:18,299 - INFO - Successfully read Demographics.csv file with shape: (11347, 6) and unique PatientIDs: 11347
2025-11-28 21:43:18,313 - INFO - Successfully processed Demographics.csv file with final shape: (6996, 6) and unique PatientIDs: 6996


In [29]:
demographics_df.Gender.value_counts(dropna = False)

Gender
M    5396
F    1600
Name: count, dtype: int64

In [30]:
# Impute missing with most common sex (male)
demographics_df['sex_male'] = np.where(demographics_df['Gender'] == 'F', 0, 1)

In [31]:
demographics_df = demographics_df.drop(columns = ['Gender'])

### Process Enhanced_AdvNSCLCBiomarkers.csv

In [32]:
biomarkers_df = processor.process_biomarkers(file_path = '../data/Enhanced_AdvHeadNeckBiomarkers.csv',
                                             index_date_df = index_date_df, 
                                             index_date_column = 'StartDate',
                                             days_before = None, 
                                             days_after = 30,
                                             pdl1_result_type = 'cps')

2025-11-28 21:43:18,367 - INFO - Successfully read Enhanced_AdvHeadNeckBiomarkers.csv file with shape: (4379, 17) and unique PatientIDs: 3131
2025-11-28 21:43:18,375 - INFO - Successfully merged Enhanced_AdvHeadNeckBiomarkers.csv df with index_date_df resulting in shape: (3108, 18) and unique PatientIDs: 2198
2025-11-28 21:43:18,377 - WARNING - Found 3 records (PatientIDs: ['F22DE615707AB', 'FA92F4315E170', 'F62450C3FE8C1']) with PD-L1 positive but CPS 0 or <1 - possible data quality issue
2025-11-28 21:43:18,429 - INFO - Successfully processed Enhanced_AdvHeadNeckBiomarkers.csv file with final shape: (6996, 3) and unique PatientIDs: 6996


In [33]:
biomarkers_df.PDL1_status.value_counts(dropna = False)

PDL1_status
NaN         5688
unknown      757
positive     376
negative     175
Name: count, dtype: int64

In [34]:
biomarkers_df['PDL1_status'] = biomarkers_df['PDL1_status'].fillna('unknown')

### Process ECOG.csv

In [35]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = index_date_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2025-11-28 21:43:18,523 - INFO - Successfully read ECOG.csv file with shape: (199312, 4) and unique PatientIDs: 8957
2025-11-28 21:43:18,573 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (148641, 5) and unique PatientIDs: 5942
2025-11-28 21:43:18,666 - INFO - Successfully processed ECOG.csv file with final shape: (6996, 3) and unique PatientIDs: 6996


In [36]:
ecog_df.ecog_index.value_counts(dropna = False)

ecog_index
1      2378
NaN    2189
0      1619
2       660
3       138
4        12
Name: count, dtype: int64

In [37]:
ecog_df['ecog_index'] = ecog_df['ecog_index'].astype('float64')
ecog_df['ecog_index_na'] = np.where(ecog_df['ecog_index'].isna(), 1, 0)

# impute 1 for missing ECOG since most common
ecog_df['ecog_index'] = ecog_df['ecog_index'].fillna(1)

In [38]:
ecog_df['ecog_newly_gte2'] = ecog_df['ecog_newly_gte2'].fillna(0)

### Process Vitals.csv

In [39]:
vitals_df = processor.process_vitals(file_path = '../data/Vitals.csv',
                                     index_date_df = index_date_df,
                                     index_date_column = 'StartDate',
                                     weight_days_before = 90,
                                     days_after = 0,
                                     vital_summary_lookback = 180, 
                                     abnormal_reading_threshold = 1)

2025-11-28 21:43:22,624 - INFO - Successfully read Vitals.csv file with shape: (4024923, 16) and unique PatientIDs: 11339
2025-11-28 21:43:24,608 - INFO - Successfully merged Vitals.csv df with index_date_df resulting in shape: (2776377, 17) and unique PatientIDs: 6996
2025-11-28 21:43:25,660 - INFO - Successfully processed Vitals.csv file with final shape: (6996, 8) and unique PatientIDs: 6996


### Process Lab.csv

In [40]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = index_date_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2025-11-28 21:43:37,610 - INFO - Successfully read Lab.csv file with shape: (9064402, 17) and unique PatientIDs: 10989
2025-11-28 21:43:41,436 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (6537558, 18) and unique PatientIDs: 6940
2025-11-28 21:43:54,068 - INFO - Successfully processed Lab.csv file with final shape: (6996, 76) and unique PatientIDs: 6996


### Process MedicationAdministration.csv

In [41]:
medications_df = processor.process_medications(file_path = '../data/MedicationAdministration.csv',
                                               index_date_df = index_date_df,
                                               index_date_column = 'StartDate',
                                               days_before = 90,
                                               days_after = 0)

2025-11-28 21:43:55,433 - INFO - Successfully read MedicationAdministration.csv file with shape: (1098540, 11) and unique PatientIDs: 9978
2025-11-28 21:43:55,838 - INFO - Successfully merged MedicationAdministration.csv df with index_date_df resulting in shape: (757699, 12) and unique PatientIDs: 6846
2025-11-28 21:43:55,898 - INFO - Successfully processed MedicationAdministration.csv file with final shape: (6996, 9) and unique PatientIDs: 6996


### Process Diagnosis.csv

In [42]:
diagnosis_df = processor.process_diagnosis(file_path = '../data/Diagnosis.csv',
                                           index_date_df = index_date_df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0)

2025-11-28 21:43:56,349 - INFO - Successfully read Diagnosis.csv file with shape: (696197, 6) and unique PatientIDs: 11347
2025-11-28 21:43:56,493 - INFO - Successfully merged Diagnosis.csv df with index_date_df resulting in shape: (459896, 7) and unique PatientIDs: 6996
2025-11-28 21:43:57,551 - INFO - Successfully processed Diagnosis.csv file with final shape: (6996, 38) and unique PatientIDs: 6996


### Process Enhanced_Mortality_V2.csv

In [43]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = index_date_df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv',
                                           drop_dates = False)

2025-11-28 21:43:57,569 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (7857, 2) and unique PatientIDs: 7857
2025-11-28 21:43:57,577 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (6996, 3) and unique PatientIDs: 6996
2025-11-28 21:43:57,947 - INFO - The following columns ['last_visit_date'] are used to calculate the last EHR date
2025-11-28 21:43:57,952 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (6996, 6) and unique PatientIDs: 6996. There are 0 out of 6996 patients with missing duration values


In [44]:
mortality_df = mortality_df[['PatientID', 'event', 'duration']]

In [45]:
mortality_df = mortality_df.query('duration >= 0')

## Merge dataframes

In [46]:
training_df = merge_dataframes(enhanced_df,
                               demographics_df,
                               biomarkers_df,
                               ecog_df,
                               vitals_df,
                               labs_df,
                               medications_df,
                               diagnosis_df,
                               mortality_df,
                               merge_type = 'inner')

2025-11-28 21:43:57,972 - INFO - Anticipated number of merges: 8
2025-11-28 21:43:57,973 - INFO - Anticipated number of columns in final dataframe presuming all columns are unique except for PatientID: 154
2025-11-28 21:43:57,974 - INFO - Dataset 1 shape: (6996, 15), unique PatientIDs: 6996
2025-11-28 21:43:57,975 - INFO - Dataset 2 shape: (6996, 6), unique PatientIDs: 6996
2025-11-28 21:43:57,977 - INFO - Dataset 3 shape: (6996, 3), unique PatientIDs: 6996
2025-11-28 21:43:57,979 - INFO - Dataset 4 shape: (6996, 4), unique PatientIDs: 6996
2025-11-28 21:43:57,980 - INFO - Dataset 5 shape: (6996, 8), unique PatientIDs: 6996
2025-11-28 21:43:57,980 - INFO - Dataset 6 shape: (6996, 76), unique PatientIDs: 6996
2025-11-28 21:43:57,982 - INFO - Dataset 7 shape: (6996, 9), unique PatientIDs: 6996
2025-11-28 21:43:57,984 - INFO - Dataset 8 shape: (6996, 38), unique PatientIDs: 6996
2025-11-28 21:43:57,985 - INFO - Dataset 9 shape: (6974, 3), unique PatientIDs: 6974
2025-11-28 21:43:57,989 - 

In [47]:
training_df.shape

(6974, 154)

## Export dataframe

In [48]:
training_df.to_csv('../outputs/1L_features_training.csv', index = False)

In [49]:
# Save dtypes
training_df.dtypes.apply(lambda x: x.name).to_csv('../outputs/1L_features_training_dtypes.csv')